# Лабораторная работа 7. Метрические и байесовские методы классификации

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 6 |
| Опора на лекции | лекция 6: обобщённый метрический классификатор (опр. 6.1), выбор параметров скользящим контролем, отступ и алгоритм STOLP (опр. 6.4), байесовское решающее правило (теорема 6.7), наивный байесовский классификатор (опр. 6.9), нормальный дискриминантный анализ, LDA (теорема 6.12); лекции 1 и 4: ММП, скользящий контроль |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 4 ч самостоятельно |

## Цель работы

Реализовать обобщённый метрический классификатор и получить из него kNN, парзеновское окно и метод потенциальных функций как частные случаи; увидеть, что метрические методы бессмысленны без масштабирования, и измерить проклятие размерности; отобрать эталоны алгоритмом STOLP; сравнить эмпирический классификатор с байесовским оптимумом там, где истинные плотности известны; понять, когда «наивное» предположение о независимости признаков работает, а когда ломается.

## Что нужно сдать

Заполненный ноутбук `lab07_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import load_digits, load_wine, make_blobs, make_classification
from sklearn.model_selection import (GridSearchCV, LeaveOneOut, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=7)
describe_variant(variant)

---
# Часть 1. Обобщённый метрический классификатор

Определение 6.1: при фиксированном объекте $x$ обучающая выборка упорядочивается
по возрастанию расстояния до него, $x_{(1)}, \dots, x_{(\ell)}$, и

$$
a(x) = \arg\max_{k}\ \Gamma_k(x), \qquad
\Gamma_k(x) = \sum_{i=1}^{\ell}\bigl[y_{(i)} = k\bigr]\, w(i, x),
$$

где $w(i, x) \ge 0$ — вес $i$-го соседа. Выбор $w$ порождает все методы семейства:

| метод | вес $w(i,x)$ |
|---|---|
| $k$ ближайших соседей | $[i \le k]$ |
| kNN с весами | $[i \le k]\,q^{\,i}$, $q \in (0,1)$ |
| парзеновское окно фиксированной ширины $h$ | $K\bigl(\rho(x, x_{(i)})/h\bigr)$ |
| парзеновское окно переменной ширины | $K\bigl(\rho(x, x_{(i)})/\rho(x, x_{(k+1)})\bigr)$ |
| потенциальные функции | $\gamma_i\,K\bigl(\rho(x, x_i)/h_i\bigr)$ |

Реализуйте **одну** функцию с параметром-весом и получите из неё все методы.
Ваш вариант: `variant["metric_method"]`, ядро `variant["kernel"]`,
метрика `variant["distance"]`.

In [ ]:
KERNELS = {
    "прямоугольное": lambda r: (np.abs(r) <= 1) * 0.5,
    "треугольное": lambda r: np.maximum(0.0, 1 - np.abs(r)),
    "Епанечникова": lambda r: np.maximum(0.0, 0.75 * (1 - r ** 2)),
    "квартическое": lambda r: np.maximum(0.0, (15 / 16) * (1 - r ** 2) ** 2),
    "гауссовское": lambda r: np.exp(-0.5 * r ** 2) / np.sqrt(2 * np.pi),
}

# TODO: реализуйте словарь DISTANCES с четырьмя метриками
#       (евклидова -- через тождество из работы 1, часть 1).

class MetricClassifier:
    """Обобщённый метрический классификатор (опр. 6.1).

    Один класс, пять методов -- различаются ТОЛЬКО функцией веса w(i, x):
      'knn'             : [i <= k]
      'knn_weighted'    : [i <= k] q^i
      'parzen_fixed'    : K(rho / h)
      'parzen_variable' : K(rho / rho_(k+1))
      'potential'       : gamma_i K(rho / h_i)
    Подсказка: номер соседа i получается двойным argsort матрицы расстояний.
    Класс должен наследоваться от sklearn.base.BaseEstimator и ClassifierMixin,
    иначе его нельзя передать в GridSearchCV/cross_val_score.
    """
    # TODO


# TODO: постройте графики всех пяти ядер на одном рисунке.

### Задание 1.2. Все методы на одной картинке

На двумерной выборке постройте границы решения для всех пяти методов при
одинаковом ядре и сравните их между собой и со `sklearn.neighbors.KNeighborsClassifier`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

Xm, ym = make_blobs(n_samples=220, centers=3, cluster_std=1.6, random_state=RANDOM_STATE)
Xm = StandardScaler().fit_transform(Xm)

# TODO: постройте границы решения всех пяти методов на одном рисунке
#       и сверьте свой kNN(5) со sklearn (совпадение предсказаний должно быть ~1.0).

> **Вывод.** Чем отличаются границы, полученные разными весовыми функциями? Что происходит с парзеновским окном фиксированной ширины в разреженной области выборки и как это лечит окно переменной ширины?
>
> *(ваш ответ здесь)*

---
# Часть 2. Масштаб признаков и проклятие размерности

Метрические методы опираются на $\rho(x, x')$, а расстояние **не инвариантно
к единицам измерения**: признак, измеренный в рублях, полностью подавит признак,
измеренный в долях. Это не тонкость реализации, а свойство самого метода.

Вторая проблема — размерность. В $\mathbb{R}^n$ при больших $n$ все попарные
расстояния становятся почти одинаковыми, и понятие «ближайший сосед» теряет смысл.
Измерьте оба эффекта.

In [ ]:
# TODO: (1) на выборке wine сравните точность kNN(5) без масштабирования и со
#           StandardScaler; выведите отношение самого большого и самого малого
#           стандартных отклонений признаков;
#       (2) проклятие размерности: для n из [1,2,3,5,10,20,50,100,300,1000]
#           сгенерируйте 500 точек из U[0,1]^n, посчитайте попарные расстояния и
#           относительный контраст (max - min)/mean. Постройте график контраста
#           и гистограммы нормированных расстояний для нескольких n.

> **Вывод.** Во сколько раз изменилась точность после масштабирования? Как ведёт себя контраст расстояний с ростом размерности и почему это убивает метрические методы? Как с этим борются?
>
> *(ваш ответ здесь)*

---
# Часть 3. Выбор гиперпараметров и эффективный LOO

Для kNN контроль по отдельным объектам считается **почти бесплатно**: достаточно
один раз отсортировать соседей, а затем для каждого объекта исключить его самого
из списка. Это в $\ell$ раз быстрее наивной реализации.

Реализуйте эффективный LOO и подберите им $k$ (или $h$ — в зависимости от вашего
метода). Сравните с наивным перебором по времени.

In [ ]:
import time


def loo_knn_fast(X, y, k_grid, distance="евклидово"):
    """LOO для kNN сразу по всей сетке k.

    Подсказка: посчитайте матрицу расстояний один раз, поставьте на диагональ
    бесконечность, отсортируйте соседей и накапливайте счётчики классов,
    добавляя по одному соседу за шаг.
    """
    raise NotImplementedError


# TODO: 1) подберите k по быстрому LOO на сетке 1..30, постройте график;
#       2) сравните по времени с наивным LOO (для 12 значений k достаточно)
#          и убедитесь, что результаты совпадают;
#       3) подберите ширину окна h для парзеновского метода по 5-кратному контролю.

---
# Часть 4. Отступ и отбор эталонов (STOLP)

Определение 6.4: отступ объекта в метрическом классификаторе —

$$
M(x_i) = \Gamma_{y_i}(x_i) - \max_{k \ne y_i}\Gamma_k(x_i).
$$

По величине отступа объекты делятся на эталонные ($M \gg 0$), неинформативные,
пограничные ($M \approx 0$), ошибочные ($M < 0$) и шумовые ($M \ll 0$).
Алгоритм STOLP отбирает небольшое подмножество эталонов, выбрасывая шум
и «неинформативную массовку».

In [ ]:
def margins(clf, X, y):
    """M(x_i) = Gamma_{y_i}(x_i) - max_{k != y_i} Gamma_k(x_i) (отступ, опр. 6.4)."""
    raise NotImplementedError


def stolp(X, y, delta=0.0, max_errors=0.01, k=5, kernel="Епанечникова"):
    """STOLP: (1) выбросить объекты с M < delta; (2) начать с одного эталона
    на класс (максимальный отступ); (3) пока ошибок больше max_errors,
    добавлять объект с наименьшим отступом."""
    raise NotImplementedError


X_st, y_st = make_blobs(n_samples=300, centers=3, cluster_std=2.2, random_state=RANDOM_STATE)
X_st = StandardScaler().fit_transform(X_st)
noise_idx = np.random.default_rng(0).choice(len(y_st), 18, replace=False)
y_st[noise_idx] = (y_st[noise_idx] + 1) % 3

# TODO: 1) примените STOLP, выведите число эталонов и сколько среди них
#          объектов с испорченными метками;
#       2) сравните точность на СВЕЖИХ данных: вся выборка против эталонов;
#       3) постройте профиль отступов и картинку с отмеченными эталонами.

> **Вывод.** Сколько объектов оставил STOLP и как изменилась точность? Попали ли испорченные метки в число эталонов? Зачем вообще отбирать эталоны, если хранить всю выборку несложно?
>
> *(ваш ответ здесь)*

---
# Часть 5. Байесовское решающее правило

Теорема 6.7: при известных априорных вероятностях $P_k$ и плотностях $p_k(x)$
минимум среднего риска даёт правило

$$
a(x) = \arg\max_k\ \lambda_k P_k\, p_k(x),
$$

где $\lambda_k$ — цена ошибки на объекте класса $k$ (при равных ценах множитель
опускается). Это **оптимальный** классификатор: ни один алгоритм не может дать
меньшую вероятность ошибки.

Уникальная возможность: если мы **сами** породили данные, то $p_k$ известны, и
байесовский оптимум вычислим точно. Сравним с ним всё, что умеем.

In [ ]:
from scipy.stats import multivariate_normal
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

MU = [np.array([0.0, 0.0]), np.array([2.0, 1.4])]
COV = [np.array([[1.0, 0.75], [0.75, 1.0]]), np.array([[1.6, -0.9], [-0.9, 1.0]])]
PRIOR = [0.6, 0.4]

# TODO: 1) напишите генератор смеси и байесовское правило теоремы 6.7
#          (плотности ИЗВЕСТНЫ -- используйте multivariate_normal.pdf);
#       2) посчитайте точность оптимума на большой выборке -- это верхняя граница
#          для всех моделей и «неустранимая ошибка» задачи;
#       3) сравните с ним наивный Байес, LDA, QDA, kNN и логистическую регрессию;
#       4) нарисуйте четыре панели с границами решения, наложив пунктиром
#          байесовскую границу.

> **Вывод.** Какая модель ближе всего подошла к байесовскому оптимуму и почему? Почему LDA даёт прямую, а QDA — кривую? Почему наивный Байес проигрывает, хотя данные действительно гауссовские?
>
> *(ваш ответ здесь)*

---
# Часть 6. Наивный байесовский классификатор: где он работает

Определение 6.9: предполагается независимость признаков внутри класса,

$$
p_k(x) = \prod_{j=1}^{n} p_{kj}(x_j)
\;\Longrightarrow\;
a(x) = \arg\max_k \Bigl(\ln P_k + \sum_{j=1}^{n}\ln p_{kj}(x_j)\Bigr).
$$

Предположение почти всегда ложно, но метод часто работает: для принятия решения
важен не точный вид апостериорных вероятностей, а лишь то, какой класс победит.
Проверим границы применимости — и разберём классическое приложение к текстам.

In [ ]:
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

# TODO: (1) для корреляции признаков внутри класса из [0, 0.3, 0.6, 0.85, 0.95, 0.99]
#           сгенерируйте двумерную смесь двух гауссиан с ОДИНАКОВОЙ ковариацией
#           и средними mu_0 = (0,0), mu_1 = (1.8, 0) -- важно, чтобы разность
#           средних не была направлена вдоль собственного вектора ковариации.
#           Сравните точность GaussianNB, LDA и QDA. Постройте график.
#       (2) примените наивный Байес к текстовой задаче
#           (fetch_20newsgroups с двумя категориями и CountVectorizer;
#           при отсутствии интернета -- к load_digits) и сравните
#           MultinomialNB, BernoulliNB и логистическую регрессию.
#           Для MultinomialNB выведите по 8 самых характерных слов каждого класса
#           (по разности feature_log_prob_).

> **Вывод.** При какой корреляции наивный Байес начинает заметно проигрывать? Почему на текстах он работает хорошо, хотя слова в предложении заведомо зависимы?
>
> *(ваш ответ здесь)*

---
# Часть 7. Своя выборка

Сравните на индивидуальной выборке метрический метод вашего варианта
(с подобранными гиперпараметрами) и байесовский метод вашего варианта
(`variant["bayes_method"]`), а также — при `variant["use_stolp"] = True` —
эффект отбора эталонов.

In [ ]:
data = load_personal(variant)
# TODO: 1) при необходимости бинаризуйте цель по медиане обучающей выборки;
#       2) подберите скользящим контролем гиперпараметр своего метрического метода
#          (k или h) и оцените качество на контроле;
#       3) обучите байесовский метод своего варианта и сравните;
#       4) если variant["use_stolp"] -- отберите эталоны и посмотрите, как
#          изменится качество и во сколько раз сократится выборка;
#       5) добавьте строку с константным ответом.

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Все пять методов части 1 — частные случаи одной формулы. Выпишите весовую функцию $w(i,x)$ для каждого и объясните, какой из них не имеет настраиваемых параметров вовсе.
2. Почему kNN обязательно требует масштабирования признаков, а решающее дерево — нет? Что в устройстве методов даёт это различие?
3. Байесовский оптимум на вашей задаче даёт точность 0.87. Ваша модель показала 0.86. Стоит ли продолжать улучшать модель?
4. Наивный байесовский классификатор предполагает независимость признаков. Назовите ситуацию, где это предположение полностью ложно, но метод всё равно даёт верный ответ, и объясните почему.
5. Чем отличаются LDA и QDA по числу оцениваемых параметров? При каком размере выборки вы предпочтёте LDA, даже зная, что ковариации классов различны?

### Домашнее задание

1. **Формула Надарая–Ватсона.** Метрический подход переносится на регрессию: $a(x) = \sum_i y_i w(i,x) \big/ \sum_i w(i,x)$. Реализуйте её со своим ядром, подберите ширину окна по LOO и сравните с `KNeighborsRegressor(weights='distance')`. Покажите на графике поведение оценки **на краях** отрезка и объясните возникающее там смещение (подсказка: у края окно несимметрично, и среднее соседей смещено внутрь области).

2. **Метод потенциальных функций.** Реализуйте обучение зарядов $\gamma_i$: изначально все нули; перебирая объекты, при ошибке на объекте $x_i$ увеличивать $\gamma_i$ на единицу; повторять до стабилизации. Покажите, что число ненулевых зарядов мало (это аналог опорных векторов из работы 4), сравните множество объектов с $\gamma_i > 0$ с эталонами STOLP из части 4 и с опорными векторами SVM на тех же данных. Совпадают ли они?